# Week 3
First some exploration to see if we can shrink the parquet without losing `user_id` signal

In [1]:
import duckdb
from datetime import datetime
import colornames
from collections import defaultdict

PARQUET_PATH = "../2022_place_canvas_history_uid.parquet"
START_TIME = datetime(int(2022), int(4), int(1), int(12))
END_TIME = datetime(int(2022), int(4), int(1), int(18))
MAX_INACTIVITY = 900

In [27]:
PARQUET_PATH = "../test.parquet"
START_TIME = datetime(int(2022), int(4), int(4), int(0))
END_TIME = datetime(int(2022), int(4), int(4), int(2))

In [2]:
query = f"""
    SELECT *
    FROM '{PARQUET_PATH}'
    WHERE
        timestamp BETWEEN '{START_TIME}' AND '{END_TIME}'
"""

filtered_df = duckdb.query(query).df()

# Rank Colors By Distinct Users
Report a ranking of colors by the number of distinct users who placed those during the specified timeframe.

In [4]:
query = f"""
    SELECT
        pixel_color,
        COUNT(DISTINCT user_id) AS user_count
    FROM
        filtered_df
    GROUP BY
        pixel_color
    ORDER BY
        user_count DESC
"""

res = duckdb.query(query)

for i, tup in enumerate(res.fetchall()):
    hex_color = tup[0]
    freq = tup[1]
    color_name = colornames.find(hex_color)

    print(f"{i+1}. {color_name}: {freq}")


1. Black: 412124
2. Vermilion: 299003
3. White: 270212
4. Azure: 244148
5. Bright Sun: 201025
6. Turquoise Blue: 117115
7. Pink Salmon: 116719
8. Screamin' Green: 102042
9. Green Haze: 95611
10. Violet Eggplant: 77678
11. Yellow Sea: 77006
12. Curious Blue: 69604
13. Iron: 38707
14. Oslo Gray: 37131
15. Kumera: 30551
16. Amethyst: 29539


# Calculate Average Session Length
A session is in progress until a 15-minute window of inactivity is reached. A user must have placed more than one pixel to have had a session.

Return the average session length in seconds during the specified timeframe.

In [11]:
query = f"""
    WITH
        time_diff AS (
            SELECT
                *,
                EPOCH(timestamp) - LAG(EPOCH(timestamp)) OVER (PARTITION BY user_id ORDER BY timestamp ASC) AS inactivity_time
            FROM
                filtered_df
            ORDER BY
                user_id, timestamp ASC
        ),
        sessions AS (
            SELECT
                SUM(
                    CASE WHEN 
                        inactivity_time > {MAX_INACTIVITY} OR inactivity_time IS NULL 
                    THEN 1 
                    ELSE 0 END
                ) OVER (PARTITION BY user_id ORDER BY timestamp ASC) AS session_id,
                user_id,
                timestamp
            FROM
                time_diff
        ),
        session_lengths AS (
            SELECT
                user_id,
                session_id,
                EPOCH(MAX(timestamp)) - EPOCH(MIN(timestamp)) AS session_length
            FROM
                sessions
            GROUP BY
                user_id, session_id
            HAVING
                COUNT(*) > 1
        )
    SELECT
        ROUND(AVG(session_length), 2) AS avg_session_length_seconds
    FROM
        session_lengths
"""

res = duckdb.query(query).fetchone()[0]

print(res)

# Jazzy got ~1639.97 seconds

1639.96


# Pixel Placement Percentiles
Calculate the 50th, 75th, 90th, and 99th percentiles of the number of pixels placed by users during the specified timeframe.

In [25]:
query = f"""
    WITH
        dist AS (
            SELECT
                user_id,
                COUNT(*) AS freq
            FROM
                filtered_df
            GROUP BY
                user_id
        )
    SELECT
        quantile_cont(freq, 0.50 ORDER BY freq ASC) AS '50_percentile',
        quantile_cont(freq, 0.75 ORDER BY freq ASC) AS '75_percentile',
        quantile_cont(freq, 0.90 ORDER BY freq ASC) AS '90_percentile',
        quantile_cont(freq, 0.99 ORDER BY freq ASC) AS '99_percentile'
    FROM
        dist
"""
p50, p75, p90, p99 = duckdb.query(query).fetchone()

print(p50)

2.0


# Count First-Time Users
Count how many users placed their first pixel ever within the specified timeframe.

In [7]:
query = f"""
    WITH
        during_timeframe AS (
            SELECT
                DISTINCT user_id
            FROM
                filtered_df
        ),
        before_timeframe AS (
            SELECT
                DISTINCT user_id
            FROM
                '{PARQUET_PATH}'
            WHERE
                timestamp < '{START_TIME}'
        )
    SELECT
        count(*) AS new_users
    FROM
        before_timeframe AS btf
    RIGHT JOIN during_timeframe AS dtf ON
        btf.user_id = dtf.user_id
    WHERE
        btf.user_id IS NULL
"""
duckdb.query(query)

┌───────────┐
│ new_users │
│   int64   │
├───────────┤
│   1062702 │
└───────────┘

In [ ]:
query = f"""
    WITH
        user_times AS (
            SELECT
                user_id,
                EPOCH(MAX(timestamp)) - EPOCH(MIN(timestamp)) AS session_length
            FROM
                filtered_df
            GROUP BY
                user_id
        ),
        session_lengths AS (
            SELECT
                user_id,
                SUM(CASE WHEN time_diff < {MAX_INACTIVITY} THEN time_diff ELSE 0 END) AS session_length
            FROM
                user_times
            GROUP BY
                user_id
            HAVING
                session_length > 0
        )
    SELECT
        AVG(session_length)
    FROM
        user_times
"""
#2445.459
duckdb.query(query)

BinderException: Binder Error: Referenced column "time_diff" not found in FROM clause!
Candidate bindings: "user_id"